In [ ]:
!pip install langchain-groq langchain-core
!pip install langchain-community langchain-groq langchain
!pip install langchain-community

In [8]:
import os
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
import os
from dotenv import load_dotenv

# سحب المفتاح من خزنة كولاب
load_dotenv()

# استخدامه في مشروعك
api_key = os.getenv("GROQ_API_KEY")
class MasryChatbot:
    def __init__(self):
        # تم تحديث الموديل لـ llama-3.3-70b-versatile عشان يتجنب خطأ الـ decommissioned
        self.llm = ChatGroq(
            model_name="llama-3.3-70b-versatile", 
            temperature=0.8
        )

        self.store = {}

        # 2. هندسة الرد المصري
        self.prompt = ChatPromptTemplate.from_messages([
            ("system", (
                "أنت مساعد ذكي ولطيف اسمك 'صاحبي'. "
                "تتحدث اللهجة العامية المصرية بطلاقة وخفة دم. "
                "ردودك لازم تكون بالمصري، ودودة، ومحترمة. "
                "لو حد سألك عن حاجة متعرفهاش، قوله بصراحة بس بلطافة."
            )),
            MessagesPlaceholder(variable_name="history"),
            ("human", "{input}"),
        ])

        # 3. بناء الهيكل
        self.chain = self.prompt | self.llm

    def get_session_history(self, session_id: str):
        if session_id not in self.store:
            self.store[session_id] = ChatMessageHistory()
        return self.store[session_id]

    def ask(self, user_input: str, session_id: str = "user_1"):
        try:
            if not user_input.strip():
                return "مبعتليش حاجة ليه يا غالي؟"

            # استخدام الكلاس ده بيضمن إن الذاكرة تفضل شغالة طول ما البرنامج مفتوح
            with_message_history = RunnableWithMessageHistory(
                self.chain,
                self.get_session_history,
                input_messages_key="input",
                history_messages_key="history",
            )

            response = with_message_history.invoke(
                {"input": user_input},
                config={"configurable": {"session_id": session_id}}
            )
            return response.content
        except Exception as e:
            # لو الـ API Key فيه مشكلة أو الموديل مش متاح هيظهرلك هنا
            if "model_decommissioned" in str(e):
                return "الموديل ده قديم، غيرتلك الموديل لواحد أحدث، جرب تاني!"
            print(f"Error details: {e}")
            return "حصلت لخبطة بسيطة، ممكن تجرب تبعت تاني؟"

# --- التشغيل ---
bot = MasryChatbot()
print("--- 'صاحبي' رجع وبقوة! (اكتب 'خروج' للإنهاء) ---")

while True:
    user_msg = input("أنت: ")
    if user_msg.lower() in ['خروج', 'exit', 'quit']:
        print("صاحبي: نورتني يا غالي، في رعاية الله!")
        break

    answer = bot.ask(user_msg)
    print(f"صاحبي: {answer}")

--- 'صاحبي' رجع وبقوة! (اكتب 'خروج' للإنهاء) ---
صاحبي: نورتني يا غالي، في رعاية الله!
